In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier

Check the top 10 rows of the dataset to see what its like

In [ ]:
!head /kaggle/input/forest-cover-type-prediction/train.csv

Since it has header, pandas default parameters on read_csv can be used.

In [ ]:
df = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
df

The dataset is perfectly balanced where each class is represented the same number of samples (2160). This means, the standard multilabel accuracy metric should be sufficient to evaluate model performance.  
If the dataset is imbalanced, other metric such as F1 or ROC should be considered instead.

In [ ]:
df['Cover_Type'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.columns

Check how many missing data in the train set.  
There is no missing data in train set

In [ ]:
df.isnull().sum()

In [ ]:
indep_cols = ['Elevation', 'Aspect', 'Slope',
       'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
       'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon',
       'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points',
       'Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3',
       'Wilderness_Area4', 'Soil_Type1', 'Soil_Type2', 'Soil_Type3',
       'Soil_Type4', 'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8',
       'Soil_Type9', 'Soil_Type10', 'Soil_Type11', 'Soil_Type12',
       'Soil_Type13', 'Soil_Type14', 'Soil_Type15', 'Soil_Type16',
       'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 'Soil_Type20',
       'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24',
       'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28',
       'Soil_Type29', 'Soil_Type30', 'Soil_Type31', 'Soil_Type32',
       'Soil_Type33', 'Soil_Type34', 'Soil_Type35', 'Soil_Type36',
       'Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40']
dep_col = 'Cover_Type'

Random state is just to make the runs consistent

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df[indep_cols],df[dep_col],test_size=0.20,random_state=499712)

Use GridSearchCV to perform hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
def grid_search(parameters):
    model = ExtraTreesClassifier()
    clf = GridSearchCV(model, parameters, cv=5)
    clf.fit(X_train, y_train)
    return clf

In [ ]:
# set this to True to run grid search and save the best params.
# set this to False prior submission.
RUN_GRID_SEARCH = False

In [ ]:
if RUN_GRID_SEARCH:
    clf_est = grid_search({
        'n_estimators': [10, 20, 50, 100, 200, 500, 1000, 1200, 1500, 1800, 1900, 2000, 2100, 3000]
    })
    df_clf_est = pd.DataFrame(clf_est.cv_results_)
    df_clf_est_top = df_clf_est[['param_n_estimators', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values(by = "rank_test_score")
    df_clf_est_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_crit = grid_search({
        'criterion' : ["gini", "entropy", "log_loss"]
    })
    df_clf_crit = pd.DataFrame(clf_crit.cv_results_)
    df_clf_crit_top = df_clf_crit[['param_criterion', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values(by = "rank_test_score")
    df_clf_crit

In [ ]:
if RUN_GRID_SEARCH:
    clf_max_depth = grid_search({
        'max_depth' : [1, 2, 5, 8, 13, 21, 34, 53, 54, 55, 89, None]
    })
    df_clf_max_depth = pd.DataFrame(clf_max_depth.cv_results_)
    df_clf_max_depth_top = df_clf_max_depth[['param_max_depth', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values(by = "rank_test_score")
    df_clf_max_depth_top

In [ ]:
def find_best_one_param(param_name, param_values):
    clf = grid_search({
        param_name : param_values
    })
    df_clf = pd.DataFrame(clf.cv_results_)
    df_clf_top = df_clf[['param_%s' % param_name, 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values(by = "rank_test_score")
    return (clf, df_clf, df_clf_top)

In [ ]:
if RUN_GRID_SEARCH:
    clf_min_samples_split, df_clf_min_samples_split, df_clf_min_samples_split_top = find_best_one_param('min_samples_split', [2, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377])
    df_clf_min_samples_split_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_min_samples_leaf, df_clf_min_samples_leaf, df_clf_min_samples_leaf_top = find_best_one_param('min_samples_leaf', [1, 2, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377])
    df_clf_min_samples_leaf_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_min_weight_fraction_leaf, df_min_weight_fraction_leaf, df_min_weight_fraction_leaf_top = find_best_one_param(
        'min_weight_fraction_leaf', [x / 10 for x in range(0, 6)])
    df_min_weight_fraction_leaf_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_max_features, df_max_features, df_max_features_top = find_best_one_param(
        'max_features', ['sqrt', 'log2', 2, 5, 8, 13, 21, 34, None])
    df_max_features_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_max_leaf_nodes, df_max_leaf_nodes, df_max_leaf_nodes_top = find_best_one_param(
        'max_leaf_nodes', [2, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, None])
    df_max_leaf_nodes_top

In [ ]:
if RUN_GRID_SEARCH:
    clf_min_impurity_decrease, df_min_impurity_decrease, df_min_impurity_decrease_top = find_best_one_param(
        'min_impurity_decrease', [x / 100 for x in range(0, 11)])
    df_min_impurity_decrease_top

In [ ]:
# n_estimators=1900 produced 76.686% on kaggle test
best_params = {
    'n_estimators': 2000,
    'criterion': 'entropy',
    'max_depth': 34,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'min_weight_fraction_leaf': 0,
    'max_features': 34,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0
}

In [ ]:
!head /kaggle/input/forest-cover-type-prediction/sampleSubmission.csv

In [ ]:
!head /kaggle/input/forest-cover-type-prediction/test.csv

In [ ]:
dtest = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')
dtest

In [ ]:
dtest.isna().sum()

In [ ]:
model = ExtraTreesClassifier(**best_params)
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
y_hat = model.predict(dtest[indep_cols])

In [ ]:
dtest_submission = pd.DataFrame({
        "Id": dtest["Id"],
        "Cover_Type": y_hat
    })
dtest_submission

In [ ]:
dtest_submission.to_csv('./submission.csv', index=False)

In [ ]:
!head submission.csv

If this notebook helps you, click upvote